# Lactanet Genetics - Complete Guided Analysis

This is the **single, clean version** of the project. Run every cell in order,
top to bottom, with **Kernel → Restart & Run All**. Do not add other loading
cells above or between these - duplicated loading cells were the cause of
earlier errors (missing provinces, `KeyError`, etc.).

**Folder needed:** place your 10 `Lactanet Genetics (XX).xlsx` files (or the
`Lactanet_Genetics_XX.xlsx` style - both work) plus `DICTIONARY_lactanet.xlsx`
inside `data/raw/`, next to this notebook.


## Step 0 - Setup

Import libraries and set up the folder path.


In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

DATA_DIR = Path("data/raw")
pd.set_option("display.max_columns", 40)
sns.set_style("whitegrid")

DATA_PULL_YEAR = 2026  # year this snapshot of data was pulled, not the current year when this notebook is read


## Step 1 - Load all provincial files

Recognizes **both** file-naming styles:
- `Lactanet Genetics (AB).xlsx`
- `Lactanet_Genetics_AB.xlsx` / `Lactanet_Genetics_AB__Copie.xlsx`

If a province is missing from the printed list below, the file is not in
`data/raw/` - check the folder before continuing.


In [ ]:
PROVINCE_NAMES = {
    "AB": "Alberta", "BC": "British Columbia", "MB": "Manitoba",
    "NB": "New Brunswick", "NL": "Newfoundland and Labrador",
    "NS": "Nova Scotia", "ON": "Ontario", "PEI": "Prince Edward Island",
    "QC": "Quebec", "SK": "Saskatchewan",
}

PATTERN_PARENS = re.compile(r"\(([A-Za-z]+)\)")
PATTERN_UNDERSCORE = re.compile(r"Lactanet[_ ]Genetics[_ ]([A-Za-z]+)", re.IGNORECASE)

# Some provinces use a longer alias with no separator, e.g. "GeneticsABT.xlsx",
# "GeneticsONT.xlsx" -- map those aliases to the standard 2-3 letter codes.
CODE_ALIASES = {"ABT": "AB", "ONT": "ON", "PI": "PEI", "QUE": "QC", "SAS": "SK", "MAN": "MB"}

def extract_province_code(file_path: Path) -> str:
    stem = file_path.stem

    match = PATTERN_PARENS.search(stem) or PATTERN_UNDERSCORE.search(stem)
    if match:
        code_ = match.group(1).upper()
    else:
        # Fallback: strip out the known filler words and whatever remains is the code
        candidate = stem
        for word in ["Lactanet", "Genetics", "Copie"]:
            candidate = re.sub(word, "", candidate, flags=re.IGNORECASE)
        code_ = candidate.strip("_ ()").upper()

    code_ = CODE_ALIASES.get(code_, code_)

    if not code_ or code_ not in PROVINCE_NAMES:
        raise ValueError(f"Unrecognized province code '{code_}' in file: {file_path.name}")
    return code_

files = sorted(
    f for f in DATA_DIR.glob("*.xlsx")
    if "lactanet" in f.name.lower()
    and "genetics" in f.name.lower()
    and not f.name.startswith("~$")
)

print(f"Found {len(files)} provincial files (expected: 10):")
for f in files:
    print(" -", f.name, "->", extract_province_code(f))


In [ ]:
frames = []
for file_path in files:
    province_code = extract_province_code(file_path)
    df_province = pd.read_excel(file_path, sheet_name=0, header=1, engine="openpyxl")
    df_province.insert(0, "Province", province_code)
    df_province.insert(1, "Province Name", PROVINCE_NAMES[province_code])
    frames.append(df_province)
    print(f"{file_path.name}: {len(df_province)} records loaded ({province_code})")

print(f"\nTotal rows loaded: {sum(len(f) for f in frames)}")

# Explicit validation, rather than silently proceeding with whatever was found.
found_codes = [extract_province_code(f) for f in files]
assert len(found_codes) == 10, f"Expected 10 provincial files, found {len(found_codes)}: {found_codes}"
assert len(set(found_codes)) == 10, f"Duplicate province code detected among source files: {found_codes}"
assert set(found_codes) == set(PROVINCE_NAMES.keys()), f"Missing or unexpected province code(s): expected {sorted(PROVINCE_NAMES.keys())}, got {sorted(set(found_codes))}"
for f, df_check in zip(files, frames):
    if len(df_check) != 400:
        print(f"WARNING: {f.name} has {len(df_check)} rows, not the expected 400")
required_columns = {
    "Identification", "Name", "Birth Year", "Act.", "GS", "LPI Code",
    "LPI", "PI", "LTI", "HWI", "RI", "MI", "EI", "Pro$",
    "Milk", "Fat", "Prot", "%F", "%P",
    "ME", "FE", "BMR", "MR", "SCS",
    "Conf", "MS", "F&L", "DS", "RP",
    "%R", "REL_PROT", "REL_CONF",
}
for f, df_check in zip(files, frames):
    missing_cols = required_columns - set(df_check.columns)
    assert not missing_cols, f"{f.name} is missing required columns: {missing_cols}"
print("All validation checks passed: 10 unique provinces, required columns present.")


## Step 2 - Sort and consolidate

Combine the 10 tables into one, sorted by **LPI** (Lifetime Performance
Index, the main overall ranking index) from highest to lowest.


In [ ]:
raw = pd.concat(frames, ignore_index=True)
consolidated = raw.sort_values("LPI", ascending=False).reset_index(drop=True)

print(f"Shape: {consolidated.shape}")
print(f"Provinces present: {sorted(consolidated['Province Name'].unique())}")
consolidated.head(10)


## Step 3 - Understand the variables

Load the dictionary that explains every column.


In [ ]:
dictionary_path = DATA_DIR / "DICTIONARY_lactanet.xlsx"
dictionary = pd.read_excel(dictionary_path, engine="openpyxl")
dictionary


## Step 4 - Clean the data

- Coerce numeric columns to actual numbers.
- Turn `Act.` / `GS` letter-codes into clear `Is Active` / `Is Genomic` flags.
- Strip whitespace from text columns.


In [ ]:
clean = consolidated.copy()

text_cols = clean.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    clean[c] = clean[c].astype("string").str.strip()

NUMERIC_COLS = [
    "LPI", "PI", "LTI", "HWI", "RI", "MI", "EI", "Pro$", "Milk", "Fat", "Prot",
    "%F", "%P", "ME", "FE", "BMR", "MR", "SCS", "Conf", "MS", "F&L", "DS", "RP",
    "%R", "REL_PROT", "REL_CONF",
]
clean[NUMERIC_COLS] = clean[NUMERIC_COLS].apply(pd.to_numeric, errors="coerce")

clean["Is Active"] = clean["Act."].fillna("").eq("A").astype(bool)
clean["Is Genomic"] = clean["GS"].fillna("").eq("G").astype(bool)
clean["LPI Code"] = clean["LPI Code"].astype(str).str.strip()

print(f"Active animals: {clean['Is Active'].sum()} / {len(clean)}")
print(f"Genomically tested animals: {clean['Is Genomic'].sum()} / {len(clean)}")
clean.head()


In [ ]:
# Explicit duplicate check: within each province, and across every pair
# of provinces. This is checked directly, not assumed.
ids = (
    clean[["Province Name", "Identification"]]
    .dropna()
    .assign(Identification=lambda x: x["Identification"].astype(str).str.strip().str.upper())
)

within_province_dupes = ids.duplicated(["Province Name", "Identification"], keep=False)
print(f"Duplicate (Province, Identification) rows within the same province file: {within_province_dupes.sum()}")

cross_province_counts = ids.groupby("Identification")["Province Name"].nunique()
shared_across_provinces = cross_province_counts[cross_province_counts > 1]
print(f"Animals appearing in more than one province's file: {len(shared_across_provinces)}")

if len(shared_across_provinces) > 0:
    overlap_detail = ids[ids["Identification"].isin(shared_across_provinces.index)]
    print(overlap_detail.groupby("Identification")["Province Name"].apply(list))

assert within_province_dupes.sum() == 0, "Found duplicate IDs within a single province file"
assert len(shared_across_provinces) == 0, "Found IDs shared across provinces, the uniqueness claim below is false for this data pull"
print("Confirmed: all animals are unique, both within and across provinces.")


### Data notes to keep in mind

- **These are the top 400 animals per province by LPI**, not a full census -
  provincial averages describe each province's *best-ranked* animals, not
  its whole population. Every average in this notebook is a **dataset
  average** (computed from these 4,000 records only) - it is not an
  official national average across all Canadian dairy cattle.
- **Every one of the 4,000 animals in this dataset is unique** - no animal
  appears in more than one province's file, and there are no duplicate IDs
  within any single province's file. Confirmed directly by the code cell
  above, not assumed.
- **This is a snapshot in time, not a static or permanent dataset.**
  Lactanet's official genetic evaluations are released periodically
  (multiple times per year), and which animals rank in each province's
  top 400 changes as new births, milk recording, classification, and
  genomic results are added. The selection criterion for inclusion in
  each provincial file is **LPI at the time of the data pull** - the same
  animal could rank in or out of a future top-400 list as new evaluations
  are released. Every figure in this notebook describes this snapshot
  specifically, not a fixed or permanent ranking.

**Official definitions:**

- **`LPI Code`**: `EBV` means the animal has an *official* published index
  for **both** production and conformation, based on its own or its
  daughters' phenotype/classification records - this can be a traditional
  EBV or a genomic EBV (GEBV). `PA` (Pedigree Average / Genomic PA) means
  the animal lacks an official index in at least one of those two areas -
  this is not limited to young heifers; a mature cow can also be `PA` if
  she's missing official production or classification data for other
  reasons.
- **`GS = G` (genotyped) does not automatically mean `LPI Code = EBV`.**
  An animal can have a genomic test on file and still be `PA` if it hasn't
  yet met the phenotype/production requirements - this is expected and
  normal, not a data quality issue.
- **`%R` is NOT a reliability measure.** It stands for **Relationship
  Percent** - the animal's genetic relationship to the breed population
  (Canada's equivalent of Expected Future Inbreeding, typical range
  15-23%), used to monitor and prevent inbreeding. `REL_PROT` and
  `REL_CONF` (separate columns) are the actual reliability measures used
  elsewhere in this analysis.
- **`LPI Code` now shows a plausible PA/EBV split in all 10 provinces**
  (checked directly) - no province needs to be excluded from PA-vs-EBV
  comparisons.
- **`Act.` = inactive** mostly matches animals born in the current year
  (2026) - likely newly registered calves not yet active in milk recording,
  not deceased animals. This is a pattern in the data, not an official
  confirmation from Lactanet.
- **The "2+ years old" filter used below** is an approximation for "should
  have had the chance to reach EBV status by now" - but per the official
  definition above, a mature animal can still legitimately be `PA` for
  reasons unrelated to age, so this filter is a reasonable proxy, not a
  guarantee.
- **`GS` (genomic tested) reflects whether a result is currently on file,
  not necessarily whether an animal was ever tested** - a recently sampled
  animal awaiting results (which can take about a month) looks identical to
  one never tested. This mainly affects the youngest birth-year cohorts.


## Step 4.1 - What information does each animal actually have?

Before comparing provinces on any trait, it matters what *kind* of evidence
is behind each animal's number. Two fields determine this:

- **`LPI Code`**: `EBV` = official index built from real production/type
  records (own or daughters'). `PA` = pedigree average only, no official
  index in at least one area.
- **`GS`**: `G` = has a DNA/genomic test on file.

Crossing them gives four genuinely different information states, not just
an age or data-quality distinction.


In [ ]:
def information_group(row):
    code_val = str(row["LPI Code"]).strip().upper()
    genomic = bool(row["Is Genomic"])
    if code_val == "PA" and not genomic:
        return "1. PA (pedigree-based)"
    if code_val == "PA" and genomic:
        return "2. GPA (pedigree + genomic)"
    if code_val == "EBV" and not genomic:
        return "3. EBV (pedigree + phenotype)"
    if code_val == "EBV" and genomic:
        return "4. GEBV (pedigree + phenotype + genomic)"
    return "UNKNOWN"

clean["Information Group"] = clean.apply(information_group, axis=1)

# Fail loudly instead of silently folding unexpected values into the
# highest-information group. If this assertion ever fails, it means a
# new or malformed LPI Code value appeared in the source files.
n_unknown = (clean["Information Group"] == "UNKNOWN").sum()
assert n_unknown == 0, f"{n_unknown} rows have an unrecognized LPI Code / GS combination"
print(f"All {len(clean)} rows classified into a known information group (0 unknown).")

group_summary = clean.groupby("Information Group", observed=True).agg(
    Count=("LPI", "size"),
    Avg_REL_PROT=("REL_PROT", "mean"),
    Avg_REL_CONF=("REL_CONF", "mean"),
).round(1)
group_summary


**Reading this table:** reliability climbs from PA-only (weakest
evidence) to EBV+G (strongest) - confirming the four groups really do carry
different amounts of information, using the dataset's own reliability
columns rather than an assumption. Notably, **G+PA animals average higher
`REL` than EBV-without-G animals** - meaning genomic predictions currently
have higher reliability than phenotype-based, non-genomic animals in this dataset. This
is a statement about precision, not about which information source
matters more in a biological or causal sense.


In [ ]:
# Direct genomic vs. non-genomic comparison on reliability, with effect size.
genomic_group = clean.loc[clean["Is Genomic"], ["REL_PROT", "REL_CONF"]].dropna()
nongenomic_group = clean.loc[~clean["Is Genomic"], ["REL_PROT", "REL_CONF"]].dropna()

def cohens_d(a, b):
    n_a, n_b = len(a), len(b)
    pooled_sd = np.sqrt(((n_a - 1) * a.var(ddof=1) + (n_b - 1) * b.var(ddof=1)) / (n_a + n_b - 2))
    return (a.mean() - b.mean()) / pooled_sd

reliability_effect_rows = []
for col in ["REL_PROT", "REL_CONF"]:
    a, b = genomic_group[col], nongenomic_group[col]
    t_stat, p_val = stats.ttest_ind(a, b, equal_var=False)
    d = cohens_d(a, b)
    reliability_effect_rows.append({
        "Reliability measure": col,
        "Mean, genomic": round(a.mean(), 1),
        "Mean, non-genomic": round(b.mean(), 1),
        "Welch t": round(t_stat, 2),
        "p-value": p_val,
        "Cohen's d": round(d, 2),
    })

pd.DataFrame(reliability_effect_rows)


**Findings:** genomic status is associated with substantially higher
reliability on both measures, with large effect sizes (Cohen's d, computed
directly above). This comparison mixes GPA and GEBV animals together (all
genomically tested animals vs. all non-tested), and doesn't separately
control for age or `LPI Code` status, so it should be read as an
association between genomic status and reliability, not an isolated,
fully controlled effect of genomic testing on its own. The comparison
below isolates the genomic effect better by comparing genomic vs.
non-genomic animals separately *within* each `LPI Code` group, so age and
evaluation status are held roughly constant within each comparison.


In [ ]:
isolated_rows = []
for lpi_code in ["PA", "EBV"]:
    subset = clean[clean["LPI Code"] == lpi_code]
    genomic_sub = subset.loc[subset["Is Genomic"], ["REL_PROT", "REL_CONF"]].dropna()
    nongenomic_sub = subset.loc[~subset["Is Genomic"], ["REL_PROT", "REL_CONF"]].dropna()
    for col in ["REL_PROT", "REL_CONF"]:
        a, b = genomic_sub[col], nongenomic_sub[col]
        if len(a) > 1 and len(b) > 1:
            t_stat, p_val = stats.ttest_ind(a, b, equal_var=False)
            d = cohens_d(a, b)
            isolated_rows.append({
                "Within LPI Code": lpi_code,
                "Reliability measure": col,
                "Mean, genomic": round(a.mean(), 1),
                "Mean, non-genomic": round(b.mean(), 1),
                "n (genomic)": len(a),
                "n (non-genomic)": len(b),
                "Cohen's d": round(d, 2),
                "p-value": p_val,
            })

pd.DataFrame(isolated_rows)


**Findings, isolated comparison:** the genomic-reliability gap remains
large even holding `LPI Code` status roughly constant (GPA vs. PA within
pedigree-only animals; GEBV vs. EBV within phenotype-based animals),
supporting genomic status as a real association with reliability beyond
just reflecting animal age or evaluation status. Age itself is not fully
isolated by this check (mature animals can still differ in age within each
group), but this is a meaningfully more controlled comparison than the
all-genomic-vs-all-non-genomic split above.


In [ ]:
information_by_province = clean.groupby(["Province Name", "Information Group"], observed=True).size().unstack(fill_value=0)
information_by_province


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
information_by_province.plot(kind="barh", stacked=True, ax=ax, color=["#B23A3A", "#C9A66B", "#8FA9A0", "#2E5B4D"])
ax.set_xlabel("Number of animals (out of 400)")
ax.set_title("Information composition by province: how much evidence backs each animal's number?")
ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


**Findings:** all 10 provinces show a believable mix across all four
groups, with the EBV+G group (the most complete information) ranging from
just 2 animals (Manitoba) to 67 (Ontario), a real difference in how much
can be confidently said about each province using evaluations supported by both genomic and phenotype-based information.


## Step 4.1.1 - Robustness check: is this gap just an age effect?

The table above mixes animals of very different ages: `EBV` and `EBV+G`
animals are always 2+ years old (confirmed directly, 0 animals with an
official phenotype-based evaluation are under 2 years), but `PA only` and
`G+PA` include large numbers of very young animals simply because they
haven't had time to accumulate phenotype records yet, not necessarily
because they were passed over. Restricting all four groups to animals 2+
years old checks whether the reliability gap survives an age-controlled
comparison.


In [ ]:
DATA_PULL_YEAR = 2026  # year this snapshot of data was pulled, not the current year when this notebook is read
clean["Age (years, approx.)"] = DATA_PULL_YEAR - clean["Birth Year"]
mature_check = clean[clean["Age (years, approx.)"] >= 2]

age_controlled = mature_check.groupby("Information Group", observed=True).agg(
    N=("REL_PROT", "size"),
    Avg_REL_PROT=("REL_PROT", "mean"),
    Avg_REL_CONF=("REL_CONF", "mean"),
).round(1)

print("Reliability by group, animals 2+ years only (age-controlled):")
age_controlled


**Findings:** the gap survives the age control. Even comparing only
animals restricted to age 2 or older, `PA only` remains far less reliable
(around 42%) than `EBV+G` (around 84%), so the earlier finding is not
simply an artifact of younger animals dragging down the PA-only average.
The sample size for mature PA-only animals is smaller than the all-ages
figure (roughly 90 vs. over 1,000), a genuine reduction worth keeping in
mind, but still large enough to support the comparison.


## Step 4.2 - Sensitivity analysis: does animal-level reliability change the picture?

**Primary result throughout this notebook: the unweighted mean of each
province's 400 animals.** The top-400 list is a real, enumerated group,
not a sample used to infer a broader population - the simple mean directly
answers "what is the average official value across the animals on this
list," which is the question provincial comparisons in this notebook ask.
This also matches standard practice in dairy genetics reporting (herd
averages and genetic trend graphs are conventionally unweighted means of
published EBVs/PTAs).

Since animals differ in how reliable their individual estimates are, this
step checks - as a **sensitivity analysis, not a replacement for the
primary result** - whether weighting each animal by its own
`REL_PROT`/`REL_CONF` would change the conclusion. This is not a standard
industry procedure for computing group means (Interbull/BLUP theory uses
reliability to combine information about the *same* animal, not to weight
*different* animals against each other in a group average) - it's an
exploratory check of robustness, and is reported as one.


In [ ]:
import numpy as np

def weighted_mean(group, value_col, weight_col):
    weights = group[weight_col].fillna(0)
    values = group[value_col]
    mask = values.notna() & (weights > 0)
    return np.average(values[mask], weights=weights[mask]) if mask.sum() > 0 else np.nan

def weighted_standard_error(group, value_col, weight_col):
    weights = group[weight_col].fillna(0)
    values = group[value_col]
    mask = values.notna() & (weights > 0)
    v, w = values[mask], weights[mask]
    wmean = np.average(v, weights=w)
    variance = np.average((v - wmean) ** 2, weights=w)
    effective_n = (w.sum() ** 2) / (w ** 2).sum() # accounts for unequal weighting
    return np.sqrt(variance / effective_n), effective_n

comparison_rows = []
for province, group in clean.groupby("Province Name", observed=True):
    ebv_g_only = group[(group["LPI Code"] == "EBV") & (group["Is Genomic"])]
    track_a_mean = ebv_g_only["Conf"].mean() if len(ebv_g_only) > 0 else np.nan
    unweighted_mean = group["Conf"].mean()
    weighted, eff_n = weighted_mean(group, "Conf", "REL_CONF"), weighted_standard_error(group, "Conf", "REL_CONF")[1]

    comparison_rows.append({
        "Province": province,
        "Unweighted mean": round(unweighted_mean, 2),
        "Reliability-weighted mean": round(weighted, 2),
        "Difference (weighted - unweighted)": round(weighted - unweighted_mean, 2),
        "GEBV-only mean (reference only)": round(track_a_mean, 2) if not np.isnan(track_a_mean) else np.nan,
        "GEBV n": len(ebv_g_only),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["Sensitive to method? (diff > 1 point)"] = comparison_df["Difference (weighted - unweighted)"].abs() > 1.0
comparison_df.sort_values("Unweighted mean", ascending=False)


**How to read this:** the reliability-weighted sensitivity analysis
did not materially change provincial mean Conformation. Every
weighted-versus-unweighted difference was smaller than 0.4 points
(Saskatchewan +0.37, the largest; Manitoba -0.02, essentially unchanged,
the smallest of any province). The GEBV-only (Track A) column is shown
only as a separate reference point - with n as low as 2 (Manitoba), it
illustrates how unstable a very small subgroup mean can be, which is
exactly why it isn't used as the primary or sensitivity estimator here,
and is a distinct issue from the weighted-mean sensitivity check, which is
stable for every province.


## Step 5 - Study-dataset and provincial averages


In [ ]:
KEY_INDICES = ["LPI", "PI", "Pro$", "Milk", "Fat", "Prot", "Conf", "%R"]

study_dataset_average = clean[KEY_INDICES].mean().round(2)
print("Study-dataset average (these 4,000 records only, not an official national figure):")
print(study_dataset_average)


In [ ]:
by_province = clean.groupby("Province Name", observed=True)[KEY_INDICES].mean().round(2)
by_province["Animal Count"] = clean.groupby("Province Name", observed=True).size()
by_province = by_province.sort_values("LPI", ascending=False)
by_province


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
by_province["LPI"].plot(kind="bar", ax=ax, color="#2E5B4D")
ax.axhline(study_dataset_average["LPI"], color="red", linestyle="--", label="Study-dataset average")
ax.set_ylabel("Average LPI")
ax.set_xlabel("Province")
ax.set_title("Average LPI by province vs. study-dataset average")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Step 6 - Genomic testing (% genotyped by province)

`GS = G` means the animal has a genomic (DNA) test on file.


In [ ]:
genomic_summary = clean.groupby("Province Name", observed=True)["Is Genomic"].agg(
    Genotyped="sum", Total="count"
)
genomic_summary["Not Genotyped"] = genomic_summary["Total"] - genomic_summary["Genotyped"]
genomic_summary["% Genotyped"] = (genomic_summary["Genotyped"] / genomic_summary["Total"] * 100).round(1)
genomic_summary["% Not Genotyped"] = (100 - genomic_summary["% Genotyped"]).round(1)
genomic_summary = genomic_summary.sort_values("% Genotyped", ascending=False)
genomic_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
provinces = genomic_summary.index
genotyped = genomic_summary["% Genotyped"]
not_genotyped = genomic_summary["% Not Genotyped"]

ax.bar(provinces, genotyped, label="Genotyped (GS = G)", color="#2E5B4D")
ax.bar(provinces, not_genotyped, bottom=genotyped, label="Not genotyped", color="#C9A66B")
for i, pct in enumerate(genotyped):
    ax.text(i, pct / 2, f"{pct}%", ha="center", va="center", color="white", fontweight="bold")

ax.set_ylabel("% of animals")
ax.set_xlabel("Province")
ax.set_title("Genotyped vs. not genotyped animals, by province")
ax.set_ylim(0, 100)
ax.legend(loc="upper right")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Step 7 - Filter animals 2+ years old

We only have birth *year*, not the exact date, so age is approximate (whole
calendar years). From here on, `mature` = animals that should already be old
enough to be in production.


In [ ]:
clean["Age (years, approx.)"] = DATA_PULL_YEAR - clean["Birth Year"]
mature = clean[clean["Age (years, approx.)"] >= 2].copy()
print(f"Animals 2+ years old: {len(mature)} of {len(clean)}")

ebv = mature[mature["LPI Code"] == "EBV"].copy()
print(f"Of those, with own EBV data: {len(ebv)}")


## Step 8 - Highest Milk, Fat, and Protein by province

Best EBV animal per province for Milk, Fat, and Protein (kg) separately,
plus %Fat/%Protein. These are identified one trait at a time rather than
as a combined score, since a combined "kg score" would understate Milk's
role if it isn't part of the score but is still discussed alongside Fat
and Protein. Using EBV-only animals keeps the comparison fair (own data,
not just pedigree).


In [ ]:
top_milk_by_province = (
    ebv.sort_values(["Province Name", "Milk"], ascending=[True, False])
    .groupby("Province Name", observed=True)
    .head(1)[["Province Name", "Name", "Milk"]]
)
top_fat_by_province = (
    ebv.sort_values(["Province Name", "Fat"], ascending=[True, False])
    .groupby("Province Name", observed=True)
    .head(1)[["Province Name", "Name", "Fat"]]
)
top_prot_by_province = (
    ebv.sort_values(["Province Name", "Prot"], ascending=[True, False])
    .groupby("Province Name", observed=True)
    .head(1)[["Province Name", "Name", "Prot"]]
)

ebv["PCT Score"] = ebv["%F"] + ebv["%P"]
top_pct_by_province = (
    ebv.sort_values(["Province Name", "PCT Score"], ascending=[True, False])
    .groupby("Province Name", observed=True)
    .head(1)[["Province Name", "Name", "%F", "%P"]]
)

print("Top Milk by province:")
print(top_milk_by_province.set_index("Province Name"))
print("\nTop Fat by province:")
print(top_fat_by_province.set_index("Province Name"))
print("\nTop Protein by province:")
print(top_prot_by_province.set_index("Province Name"))
print("\nTop % components by province:")
print(top_pct_by_province.set_index("Province Name"))


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 9))

kg_compare = pd.DataFrame({
    "Milk": top_milk_by_province.set_index("Province Name")["Milk"],
    "Fat": top_fat_by_province.set_index("Province Name")["Fat"],
    "Prot": top_prot_by_province.set_index("Province Name")["Prot"],
})
kg_compare.plot(kind="bar", ax=ax1, color=["#8FA9A0", "#C9A66B", "#2E5B4D"])
ax1.set_title("Best EBV animal per province - Milk, Fat, Protein (kg), identified separately per trait")
ax1.set_ylabel("kg")
ax1.tick_params(axis="x", rotation=45)

top_pct_by_province.set_index("Province Name")[["%F", "%P"]].plot(kind="bar", ax=ax2, color=["#C9A66B", "#2E5B4D"])
ax2.set_title("Best EBV animal per province - %Fat, %Protein")
ax2.set_ylabel("%")
ax2.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


**Findings:** Kg leaders and % leaders are often *different animals* - a
cow can lead in yield (kg) without leading in concentration (%). Some top
animals in different provinces show similar names or values, but similarity
alone does not prove they are the same animal, this dataset's explicit
duplicate check (Step 4) found no shared `Identification` values across
provinces in this snapshot, so any resemblance here is coincidental or
reflects related, but distinct, animals.


## Step 9 - Best conformation animal by province


In [ ]:
top_conf_by_province = (
    ebv.sort_values(["Province Name", "Conf"], ascending=[True, False])
    .groupby("Province Name", observed=True)
    .head(1)[["Province Name", "Name", "Conf", "MS", "F&L", "DS", "RP"]]
    .set_index("Province Name")
)
top_conf_by_province


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top_conf_by_province["Conf"].sort_values(ascending=False).plot(kind="bar", ax=ax, color="#2E5B4D")
ax.set_title("Best conformation score (Conf) by province - EBV animals")
ax.set_ylabel("Conformation score")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## Step 10 - Strengths and weaknesses by province (conformation)

Green bar = above the study-dataset average for that trait (strength).
Red bar = below the study-dataset average (weakness). No heatmap - the province
name sits right on the bar so it's readable at a glance.


In [ ]:
CONFORMATION = ["Conf", "MS", "F&L", "DS", "RP"]

conf_by_province = ebv.groupby("Province Name", observed=True)[CONFORMATION].mean()
dataset_avg_conf = ebv[CONFORMATION].mean()
deviation_from_dataset_avg = conf_by_province - dataset_avg_conf

fig, axes = plt.subplots(1, 5, figsize=(22, 6))
for ax, trait in zip(axes, CONFORMATION):
    data = deviation_from_dataset_avg[trait].sort_values()
    colors = ["#B23A3A" if v < 0 else "#2E5B4D" for v in data]
    ax.barh(data.index, data.values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(trait)
    ax.set_xlabel("vs. study-dataset average")

plt.suptitle("Conformation strengths (green) and weaknesses (red) by province - EBV animals", y=1.03)
plt.tight_layout()
plt.show()


**Findings:** Quebec, Alberta, and Ontario are consistently above the
study-dataset average across all five conformation traits. Prince Edward Island is
below average on every trait - the weakest overall conformation profile.
Newfoundland and Labrador is especially weak in F&L (feet and legs).


## Step 11 - Correlation between production/components and conformation

Instead of a heatmap, this is a single sorted bar chart of every
component-vs-trait pair. Dashed gray lines mark ±0.5, the usual threshold for
"worth reporting."


In [ ]:
COMPONENT_KG = ["Milk", "Fat", "Prot"]
COMPONENT_PCT = ["%F", "%P"]

corr = ebv[COMPONENT_KG + COMPONENT_PCT + CONFORMATION].corr()

pairs = []
for comp in COMPONENT_KG + COMPONENT_PCT:
    for trait in CONFORMATION:
        pairs.append({"Pair": f"{comp} vs. {trait}", "Correlation": round(corr.loc[comp, trait], 2)})
pairs_df = pd.DataFrame(pairs).sort_values("Correlation", key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(9, 10))
colors = ["#B23A3A" if v < 0 else "#2E5B4D" for v in pairs_df["Correlation"]]
ax.barh(pairs_df["Pair"], pairs_df["Correlation"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(-0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Correlation coefficient (r)")
ax.set_title("Correlation: production/components vs. conformation traits")
plt.tight_layout()
plt.show()


In [ ]:
# Stability check: does the strongest pair hold up across random halves of the data?
def check_stability(df, col_a, col_b, n_splits=5):
    results = []
    for i in range(n_splits):
        sample = df.sample(frac=0.5, random_state=i)
        r = sample[[col_a, col_b]].corr().iloc[0, 1]
        results.append(round(r, 3))
    return results

strongest = pairs_df.iloc[-1]
comp, trait = strongest["Pair"].split(" vs. ")
print(f"Strongest pair found: {comp} vs. {trait} (r = {strongest['Correlation']})")
print(f"Across 5 random halves: {check_stability(ebv, comp, trait)}")

pair_df = ebv[[comp, trait]].dropna()
r_stat, p_value = stats.pearsonr(pair_df[comp], pair_df[trait])
print(f"Pearson r = {r_stat:.3f}, p-value = {p_value:.4f}")


**Findings:** none of the correlations cross the ±0.5 "worth reporting"
threshold. The strongest was Milk vs. Dairy Strength (DS) at only r ≈ 0.26 -
weak. **Conclusion: production traits and conformation traits show weak
linear association in this dataset** - low correlation doesn't prove
statistical or biological independence, and the top-400-by-LPI selection
restricts the range of both trait groups, which can itself reduce observed
correlations. This pattern is nonetheless consistent with how these
indices are deliberately designed in genetic evaluation systems, so that
selecting for production doesn't sacrifice structure, and vice versa. With ~1,900
animals, even tiny correlations can show a "significant" p-value - that's
why the magnitude (r) matters more than the p-value here.


## Step 12 - Conformation priorities/trends by province

Which conformation trait does each province stand out in the most, relative
to the other provinces?


In [ ]:
z_scores = (conf_by_province - conf_by_province.mean()) / conf_by_province.std()
standout_trait = z_scores.idxmax(axis=1)
standout_strength = z_scores.max(axis=1).round(2)

priority_table = pd.DataFrame({
    "Standout Trait": standout_trait,
    "Z-score": standout_strength,
}).sort_values("Z-score", ascending=False)
priority_table


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#2E5B4D" if z > 0 else "#B23A3A" for z in priority_table["Z-score"]]
ax.barh(priority_table.index, priority_table["Z-score"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
for i, (trait, z) in enumerate(zip(priority_table["Standout Trait"], priority_table["Z-score"])):
    ax.text(z, i, f" {trait}", va="center", fontsize=9)
ax.set_xlabel("How far above/below the province's own average (z-score)")
ax.set_title("Each province's standout conformation trait")
plt.tight_layout()
plt.show()


**Findings:** Alberta, Ontario, and Saskatchewan stand out most in MS
(Mammary System). Quebec stands out most in F&L (Feet & Legs) - the most
trustworthy signal here given its larger sample size. No single trait
is strongest within this study dataset; treat this as a directional signal, not a confirmed
regional breeding strategy, given the small sample sizes per province.


## Step 12.5 - Exploratory type-versus-health/reproduction profile contrast, by province

A different composite from Step 12: this contrasts type traits (`MS`,
`F&L`, `DS`, `RP`) against health and reproduction traits (`HWI`, `RI`),
both standardized so they're directly comparable, to see which side of
this contrast each province's top-400 leans toward. `Conf` is excluded
from the type side because it is itself a summary built from `MS`/`F&L`/
`DS`/`RP`, including it alongside its own components would double-count
conformation. `MI` (Milkability Index) and `EI` (Environmental Impact
Index) are excluded from the other side because neither is a health or
reproduction measure. This uses each province's full 400-animal group
(not filtered by `LPI Code`), so all 10 provinces are equally usable here.
This is an exploratory contrast, not evidence of a deliberate breeding
priority or strategy.


In [ ]:
TYPE_TRAITS = ["MS", "F&L", "DS", "RP"]
HEALTH_REPRODUCTION = ["HWI", "RI"]

type_by_province = clean.groupby("Province Name", observed=True)[TYPE_TRAITS].mean()
type_z = (type_by_province - type_by_province.mean()) / type_by_province.std()

health_repro_by_province = clean.groupby("Province Name", observed=True)[HEALTH_REPRODUCTION].mean()
health_repro_z = (health_repro_by_province - health_repro_by_province.mean()) / health_repro_by_province.std()

contrast_score = (type_z.mean(axis=1) - health_repro_z.mean(axis=1)).round(2).sort_values(ascending=False)
contrast_score.name = "Type (+) vs. Health/Reproduction (-)"
contrast_score.to_frame()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#2E5B4D" if v > 0 else "#B23A3A" for v in contrast_score]
ax.barh(contrast_score.index, contrast_score.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Leans toward Type (+) vs. Health/Reproduction (-)")
ax.set_title("Exploratory type-versus-health/reproduction profile contrast, all 10 provinces")
plt.tight_layout()
plt.show()


**Reading this:** every province is included, ranked on one
comparable scale. New Brunswick shows the highest positive score on this
exploratory contrast, followed by Nova Scotia and Ontario. Manitoba shows
the most negative score on this contrast, leaning hardest toward
health/reproduction traits of any province. This score locates each
province's top-400 profile relative to the other provinces; it does not
demonstrate a deliberate breeding strategy.


## Step 13 - Does having an EBV actually matter? (PA vs. EBV, 2+ years)

All 10 provinces now show a plausible `LPI Code` split (see the data notes in Step 4), so no provinces need to be excluded from this comparison.


In [ ]:
VALID_PROVINCE_CODES = ["AB", "BC", "MB", "NB", "NL", "NS", "ON", "PEI", "QC", "SK"]

available_codes = [c for c in VALID_PROVINCE_CODES if c in mature["Province"].unique()]
missing = set(VALID_PROVINCE_CODES) - set(available_codes)
if missing:
    print(f"Warning: these province codes are not in your data: {missing}")

valid = mature[mature["Province"].isin(available_codes)]
print(f"Rows in valid subset: {len(valid)}")
print(f"LPI Code groups found: {sorted(valid['LPI Code'].unique())}")


In [ ]:
KEY_TRAITS = ["LPI", "Pro$", "Milk", "Fat", "Prot", "Conf"]
comparison = valid.groupby("LPI Code")[KEY_TRAITS].mean().round(1)

if "EBV" in comparison.index and "PA" in comparison.index:
    comparison.loc["Difference (EBV - PA)"] = (comparison.loc["EBV"] - comparison.loc["PA"]).round(1)
    print(f"Sample sizes -> EBV: {(valid['LPI Code']=='EBV').sum()}, PA: {(valid['LPI Code']=='PA').sum()}")
else:
    print("WARNING: one of the groups (EBV or PA) is missing from 'valid' - check Step 1 for missing files.")

comparison


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
comparison.loc[["EBV", "PA"]].T.plot(kind="bar", ax=ax, color=["#2E5B4D", "#C9A66B"])
ax.set_title("EBV vs. PA animals (2+ years, reliable provinces only)")
ax.set_ylabel("Average value")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
def cohens_d(a, b):
    n_a, n_b = len(a), len(b)
    pooled_sd = np.sqrt(((n_a - 1) * a.var(ddof=1) + (n_b - 1) * b.var(ddof=1)) / (n_a + n_b - 2))
    return (a.mean() - b.mean()) / pooled_sd

effect_rows = []
for trait in KEY_TRAITS:
    a = valid.loc[valid["LPI Code"] == "EBV", trait].dropna()
    b = valid.loc[valid["LPI Code"] == "PA", trait].dropna()
    t_stat, p_val = stats.ttest_ind(a, b, equal_var=False)  # Welch's t-test, does not assume equal variances
    d = cohens_d(a, b)
    effect_rows.append({"Trait": trait, "Welch t": round(t_stat, 2), "p-value": p_val, "Cohen's d": round(d, 3), "n (EBV)": len(a), "n (PA)": len(b)})

pd.DataFrame(effect_rows)


**Findings:** differences between mature (2+ years) EBV and PA
animals were small across the board, confirmed with Welch's t-test (which
does not assume equal variances between the groups). **Only Conformation
reached statistical significance** (d=0.22, p<0.001); LPI, Pro$, Milk, Fat,
and Protein did not (p>0.05 for all, effect sizes d=0.03-0.12). Pro$ was
even slightly lower, not higher, in EBV animals, though not significantly
so. This is a much more modest result than an all-traits-significant
comparison would suggest, and doesn't support a strong claim that EBV
status confers a broad performance advantage over PA in this dataset. It's
a correlational comparison in any case, not a controlled experiment, it
may partly reflect that farmers collect full records on animals they
already expect to perform well.


## Step 14 - Correlation assumptions: are we allowed to trust Pearson's r?

Pearson's r assumes roughly normal data, a linear relationship, and no
outliers dominating the result. Checked here instead of assumed.


In [ ]:
for col in ["LPI", "Milk", "Fat", "Prot", "Conf"]:
    stat, p = stats.normaltest(clean[col].dropna())
    skew = stats.skew(clean[col].dropna())
    kurt = stats.kurtosis(clean[col].dropna())
    print(f"{col}: normaltest p={p:.4g}, skew={skew:.2f}, kurtosis={kurt:.2f}")

print()
for col in ["LPI", "Milk", "Fat", "Prot"]:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((clean[col] < lower) | (clean[col] > upper)).sum()
    print(f"{col}: {n_out} outliers ({n_out/len(clean)*100:.1f}%)")


**Findings:** formal normality tests reject normality (p<0.001) for
every trait checked, but this is expected with ~4,000 observations, where
even trivial deviations become "significant." Skew (-0.27 to 0.04) and
kurtosis (-0.80 to 0.22) are both small, and outlier share is low (0-1%) -
Pearson's r is reliable enough to use here.


## Step 15 - Regression diagnostics: VIF, residuals, homoscedasticity

This step both runs a standardized regression of LPI on its component
traits and validates it. Important context before reading the results:
per Lactanet's own published LPI structure ("Which Index is Right for My
Herd?", Lactanet, March 2025), LPI is a weighted sum of six subindexes
(Production Index 40%, Longevity and Type Index 32%, Health and Welfare
Index 8%, Reproduction Index 10%, Milkability Index 5%, Environmental
Impact Index 5%). Most traits used below are either direct components of
those subindexes or closely related published traits: `PI` is directly
composed of Fat Yield (60%) and Protein Yield (40%), with `Milk` a
correlated trait rather than a direct ingredient; `LTI`'s five direct
components are Herd Life, Mammary System, Feet & Legs, Dairy Strength, and
Rump, with overall `Conf` a correlated trait rather than one of the five.
The regression is therefore structurally linked to how LPI is constructed,
and should not be read as an independent discovery of its determinants.


In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

TRAITS = ["Milk", "Fat", "Prot", "Conf", "MS", "F&L", "DS", "RP", "HWI", "RI", "MI", "EI", "SCS"]
reg_df = clean.dropna(subset=TRAITS + ["LPI"]).copy()
X_std = (reg_df[TRAITS] - reg_df[TRAITS].mean()) / reg_df[TRAITS].std()
y_std = (reg_df["LPI"] - reg_df["LPI"].mean()) / reg_df["LPI"].std()
X_const = sm.add_constant(X_std)

model = sm.OLS(y_std, X_const).fit()
print(f"R-squared: {model.rsquared:.3f}")

vif_data = pd.DataFrame({
    "Trait": X_std.columns,
    "VIF": [variance_inflation_factor(X_std.values, i) for i in range(X_std.shape[1])],
}).sort_values("VIF", ascending=False)
print(vif_data)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.scatter(model.fittedvalues, model.resid, alpha=0.3, s=10, color="#2E5B4D")
ax1.axhline(0, color="red", linestyle="--")
ax1.set_xlabel("Fitted values")
ax1.set_ylabel("Residuals")
ax1.set_title("Residual plot")

sm.qqplot(model.resid, line="45", fit=True, ax=ax2)
ax2.set_title("QQ plot of residuals")

plt.tight_layout()
plt.show()

bp_stat, bp_p, _, _ = het_breuschpagan(model.resid, model.model.exog)
stat, p_norm = stats.normaltest(model.resid)
print(f"Breusch-Pagan (homoscedasticity): p={bp_p:.4g}")
print(f"Residual normality test: p={p_norm:.4g}, skew={stats.skew(model.resid):.2f}")


**Findings:** R²=0.961 (printed above), which given the subindex
context above should be read as confirming that the model is structurally
linked to how LPI is constructed, not as an independent discovery.
**VIF for `Conf` and `MS` is severe** (printed above) - expected, since
`Conf` is itself built from `MS`/`F&L`/`DS`/`RP` by Lactanet's own scoring,
making it mathematically redundant with them, not an independent
predictor. Breusch-Pagan confirms heteroscedasticity (p<0.001), so the
coefficients above should not be trusted with their default standard
errors. The corrected model below drops `Conf` and refits with
HC3-robust standard errors.


In [ ]:
# Corrected model: drop Conf (redundant with MS/F&L/DS/RP) and refit
# with HC3-robust standard errors to address the heteroscedasticity found above.
predictors_final = ["Milk", "Fat", "Prot", "MS", "F&L", "DS", "RP", "HWI", "RI", "MI", "EI"]
reg_df_final = clean.dropna(subset=predictors_final + ["LPI"]).copy()
X_final = (reg_df_final[predictors_final] - reg_df_final[predictors_final].mean()) / reg_df_final[predictors_final].std()
y_final = (reg_df_final["LPI"] - reg_df_final["LPI"].mean()) / reg_df_final["LPI"].std()
X_final_const = sm.add_constant(X_final)

model_hc3 = sm.OLS(y_final, X_final_const).fit(cov_type="HC3")
print(model_hc3.summary())


In [ ]:
vif_final = pd.DataFrame({
    "Trait": X_final.columns,
    "VIF": [variance_inflation_factor(X_final.values, i) for i in range(X_final.shape[1])],
}).sort_values("VIF", ascending=False)
print("VIF, corrected model (Conf dropped):")
print(vif_final)
print()
print(f"R-squared, corrected model: {model_hc3.rsquared:.3f}")


**Findings, corrected model:** with `Conf` dropped, VIF for the
remaining traits falls to a reasonable range (all below 4), confirming
`Conf` was the main source of multicollinearity. R² barely changes, since
`Conf` carried little information beyond its own components. The
HC3-robust standard errors and p-values above are the ones that should be
cited from this regression, not the uncorrected ones from the initial
model. One coefficient needs a specific correction: Milk's standardized
coefficient is close to zero in both models, but Lactanet's own published
correlation between Milk Yield and LPI is a real, moderate 0.43. The
near-zero coefficient reflects multicollinearity between Milk, Fat, and
Protein (`PI`, the Production Index, is directly composed of Fat Yield and
Protein Yield; Milk is a closely correlated trait, not a direct `PI`
ingredient), not evidence that milk volume is unimportant to LPI.


## Step 16 - Welch ANOVA + Games-Howell across all 10 provinces (classic ANOVA + Tukey HSD as sensitivity analyses)

A t-test compares 2 groups; comparing all 10 provinces at once calls for
ANOVA (does *any* province differ) followed by Tukey HSD (*which* pairs
differ, without inflating false positives from 45 separate tests). ANOVA
and Tukey both assume comparable variances across groups, checked
directly below with Levene's test rather than assumed.


In [ ]:
groups_lpi = [g["LPI"].dropna().values for _, g in clean.groupby("Province Name", observed=True)]
levene_stat, levene_p = stats.levene(*groups_lpi)
print(f"Levene's test for equal variances (LPI across provinces): stat={levene_stat:.2f}, p={levene_p:.4f}")
print("If p < 0.05, variances differ meaningfully across provinces and Welch ANOVA / Games-Howell would be more appropriate than classic ANOVA / Tukey HSD.")


In [ ]:
anova_results = []
for trait in ["LPI", "Milk", "Fat", "Prot", "Conf"]:
    groups = [g[trait].dropna().values for _, g in clean.groupby("Province Name", observed=True)]
    f_stat, p_value = stats.f_oneway(*groups)
    anova_results.append({"Trait": trait, "F-statistic": round(f_stat, 2), "p-value": p_value})

pd.DataFrame(anova_results)


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(clean["LPI"], clean["Province Name"], alpha=0.05)
tukey_df = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
print(f"Classic Tukey HSD, pairs NOT significantly different (out of {len(tukey_df)} total):")
tukey_df[tukey_df["reject"] == False]


In [ ]:
# Levene's test found unequal variances across provinces, so Tukey HSD's
# equal-variance assumption is not well met. Welch ANOVA and Games-Howell
# are the statistically appropriate alternatives here, run directly rather
# than only noting they "would be preferable."
from statsmodels.stats.oneway import anova_oneway
import pingouin as pg

welch_result = anova_oneway(clean["LPI"], groups=clean["Province Name"], use_var="unequal")
print("Welch ANOVA (does not assume equal variances):")
print(f"F={welch_result.statistic:.2f}, df=({welch_result.df_num:.0f}, {welch_result.df_denom:.1f}), p={welch_result.pvalue:.4g}")


In [ ]:
games_howell = pg.pairwise_gameshowell(data=clean, dv="LPI", between="Province Name")
not_sig_gh = games_howell[games_howell["pval"] > 0.05]
print(f"Games-Howell, pairs NOT significantly different (out of {len(games_howell)} total):")
not_sig_gh[["A", "B", "pval"]]


**Findings:** the omnibus test rejects equality of provincial means
under its assumptions for every trait checked, both classic ANOVA (p<0.001)
and Welch ANOVA, which doesn't assume equal variances (F=1373.2,
p<0.0001). **Levene's test is significant (p<0.001): variances differ
meaningfully across provinces**, so Tukey HSD's equal-variance assumption
is not well met, Games-Howell is the statistically appropriate pairwise
method here. **Games-Howell confirms the same 3 pairs found by classic
Tukey HSD are not significantly different: Alberta vs. Nova Scotia, New
Brunswick vs. Newfoundland & Labrador, and Prince Edward Island vs.
Saskatchewan.** Because the more rigorous method agrees with the simpler
one here, these 3 pairs can be reported with reasonable confidence for
this specific data snapshot, though which pairs land in this set has
changed each time a source file was corrected during this project and
should not be treated as a permanently fixed fact about these provinces.


## Step 17 - Confidence intervals (95% CI) for average LPI by province

These intervals describe the observed spread of LPI within each
province's actual top-400 list. They should not be over-interpreted as
formal sampling-uncertainty statements about a broader population: these
400 animals per province are not a random sample of a larger population,
they were selected specifically for having the highest LPI, and may
include related animals or animals sharing a herd. Narrow intervals here
mean the top-400 list itself is internally consistent, not that a
different random draw would necessarily look the same.


In [ ]:
def confidence_interval_95(series):
    n = len(series)
    mean = series.mean()
    sem = series.std() / (n ** 0.5)
    margin = sem * stats.t.ppf(0.975, n - 1)
    return mean, mean - margin, mean + margin

ci_results = []
for province, group in clean.groupby("Province Name", observed=True):
    mean, lower, upper = confidence_interval_95(group["LPI"])
    ci_results.append({"Province": province, "Mean LPI": round(mean, 1),
                        "95% CI Lower": round(lower, 1), "95% CI Upper": round(upper, 1)})

ci_df = pd.DataFrame(ci_results).sort_values("Mean LPI", ascending=False)
ci_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(ci_df))
ax.errorbar(ci_df["Mean LPI"], y_pos,
            xerr=[ci_df["Mean LPI"] - ci_df["95% CI Lower"], ci_df["95% CI Upper"] - ci_df["Mean LPI"]],
            fmt="o", color="#2E5B4D", capsize=4)
ax.set_yticks(y_pos)
ax.set_yticklabels(ci_df["Province"])
ax.set_xlabel("Mean LPI (95% CI)")
ax.set_title("Average LPI by province with 95% confidence intervals")
plt.tight_layout()
plt.show()


## Step 17.5 - What does conformation actually correlate with?

Section 16's correlation matrix already showed conformation traits barely
relate to production. This step checks the natural follow-up question:
what *do* they relate to? Three targets are tested, each with a different
relationship to conformation by design:

- `LTI` (Longevity & Type Index): per Lactanet's own published description,
  this subindex is explicitly built to combine Herd Life with Mammary
  System, Feet & Legs, Dairy Strength, and Rump. Conformation traits are
  literal ingredients here, so a strong correlation is expected by
  construction, not an independent finding.
- `HWI` (Health & Welfare Index) and `RI` (Reproduction Index): neither
  includes any conformation trait as an ingredient, so a real correlation
  here would be a genuine, non-definitional relationship.


In [ ]:
CONFORM_TRAITS = ["Conf", "MS", "F&L", "DS", "RP"]
CHECK_TARGETS = ["LTI", "HWI", "RI", "SCS", "BMR", "MI"]

conform_results = []
for c in CONFORM_TRAITS:
    for t in CHECK_TARGETS:
        sub = clean.dropna(subset=[c, t])
        r, p = stats.pearsonr(sub[c], sub[t])
        conform_results.append({"Conformation trait": c, "vs.": t, "r": round(r, 3), "p": round(p, 4), "n": len(sub)})

conform_results_df = pd.DataFrame(conform_results)
conform_results_df.reindex(conform_results_df["r"].abs().sort_values(ascending=False).index)


In [ ]:
# Stability check (5 random 50% subsamples) on the pairs that cross the
# 0.3 reporting threshold, to confirm they are not resampling noise.
def stability_check(a, b, n=5):
    sub_full = clean.dropna(subset=[a, b])
    vals = []
    for i in range(n):
        sample = sub_full.sample(frac=0.5, random_state=i)
        r, _ = stats.pearsonr(sample[a], sample[b])
        vals.append(round(r, 3))
    return vals

for a, b in [("DS", "HWI"), ("DS", "RI"), ("Conf", "HWI"), ("Conf", "RI"), ("MS", "HWI"), ("DS", "BMR"), ("MS", "SCS")]:
    print(f"{a} vs {b}: {stability_check(a, b)}")


**Findings:** `Conf`, `MS`, and `F&L` correlate strongly with `LTI`
(r=0.79-0.88), confirming the known formula structure rather than
revealing something new, since conformation traits are literal ingredients
of `LTI`.

**An observed, non-definitional negative association appears within this selected dataset with `DS` (Dairy Strength):** it
shows this with both `HWI` (r=-0.40) and
`RI` (r=-0.46), stable across resamples. Neither subindex includes `DS` as
an ingredient, so this is not definitional, though since this dataset selects only high-LPI animals (itself built partly from these subindexes), part of the association may be induced by that selection rather than reflecting an unrestricted-population effect (see Section 14 of the methodology document). Overall `Conf` shows a similar
but weaker pattern (`HWI` r=-0.25, `RI` r=-0.33). The direction is
biologically consistent with published evidence: Alcantara et al. (2022) found Body Depth (a Dairy Strength
component) associated with lower Pro$, and cited studies showing Body
Depth unfavorably correlated with productive life, number of lactations,
fertility measures, and non-return rates in Holstein cattle, though that published research does not independently reproduce these exact correlations. The same
paper found a different Dairy Strength component, Dairy Capacity,
associated *positively* with longevity, so the aggregated `DS` trait used
here may combine sub-components pulling in different directions, which
cannot be separated with the columns available in this dataset.

**The intuitive idea that better udder conformation (`MS`) means better
udder health specifically is not clearly supported:** `MS` vs. `SCS` (the
mastitis indicator) is r=-0.03, stably near zero. `MS` vs. `HWI` is weak
but consistently negative (r=-0.13), smaller than the `DS` relationships
above.

**`DS` vs. `BMR` (r=-0.61)** remains the strongest single correlation
found, but is a mechanical consequence of body size and feed maintenance
cost (see Section 4.2 of the methodology document for the full
explanation), not a health or reproduction finding like the `DS`-`HWI`/`RI`
pattern above.


## Step 18 - PCA: how do provinces group by overall genetic profile?

An earlier version of this analysis ran PCA on roughly 17 variables,
including `LPI` and `Pro$` (which are themselves weighted sums of the six
subindexes), the six subindexes, and several of the individual traits that
build those same subindexes (`Milk`/`Fat`/`Prot` build `PI`; `Conf`/`MS`/
`F&L`/`DS`/`RP` build `LTI`). Including a result alongside its own
ingredients creates circular redundancy and can make the first component
look like "everything moves together" partly because several versions of
the same construct were entered at once, not purely because of real
structure. This version uses only the six official, non-overlapping LPI
subindexes (`PI`, `LTI`, `HWI`, `RI`, `MI`, `EI`) as the province profile.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

RANDOM_SEED = 42 # fixed for reproducibility across every random process below

SUBINDEX_TRAITS = ["PI", "LTI", "HWI", "RI", "MI", "EI"]
province_profiles = clean.groupby("Province Name", observed=True)[SUBINDEX_TRAITS].mean()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(province_profiles)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
pcs = pca.fit_transform(X_scaled)
print(f"PC1: {pca.explained_variance_ratio_[0]*100:.1f}% of variance, PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")

loadings = pd.DataFrame(pca.components_.T, index=SUBINDEX_TRAITS, columns=["PC1", "PC2"])
print("\nLoadings (which subindexes drive each axis):")
print(loadings.round(3))

pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"], index=province_profiles.index)

km = KMeans(n_clusters=3, random_state=RANDOM_SEED, n_init=100)
pca_df["KMeans"] = km.fit_predict(X_scaled)
pca_df.sort_values("KMeans")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors_map = {0: "#2E5B4D", 1: "#C9A66B", 2: "#B23A3A"}
for cluster_id in sorted(pca_df["KMeans"].unique()):
    subset = pca_df[pca_df["KMeans"] == cluster_id]
    ax.scatter(subset["PC1"], subset["PC2"], s=200, color=colors_map[cluster_id], label=f"Cluster {cluster_id}")
    for name, row in subset.iterrows():
        ax.annotate(name, (row["PC1"], row["PC2"]), fontsize=9, xytext=(5, 5), textcoords="offset points")

ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%) - overall strength across all 6 subindexes")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%) - Longevity & Type vs. Environmental Impact contrast")
ax.set_title("PCA on the 6 official LPI subindexes, by province")
ax.legend()
plt.tight_layout()
plt.show()


## Step 19 - Hierarchical clustering, silhouette scores, and agreement between methods

This step cross-checks Step 18's KMeans grouping with an independent
method (hierarchical/Ward clustering), and instead of assuming `k=3` is
the right number of groups, checks the silhouette score across a range of
`k` values and reports how much the two clustering methods actually agree
(Adjusted Rand Index), rather than asserting agreement.


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.metrics import adjusted_rand_score, silhouette_score

Z = linkage(X_scaled, method="ward")

fig, ax = plt.subplots(figsize=(11, 6))
dendrogram(Z, labels=province_profiles.index.tolist(), ax=ax, color_threshold=6)
ax.set_ylabel("Distance (Ward's method)")
ax.set_title("Hierarchical clustering of provinces, 6 LPI subindexes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
pca_df["Hierarchical"] = fcluster(Z, t=3, criterion="maxclust")
ari = adjusted_rand_score(pca_df["KMeans"], pca_df["Hierarchical"])
print(f"Adjusted Rand Index, KMeans vs. hierarchical (both k=3): {ari:.3f}")
print("(1.0 = perfect agreement, 0.0 = no better than random)")
print()
print(pca_df[["KMeans", "Hierarchical"]].sort_values("KMeans"))


In [ ]:
print("Silhouette score by number of clusters (higher is better-separated):")
for k in [2, 3, 4, 5]:
    km_k = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=100).fit(X_scaled)
    sil = silhouette_score(X_scaled, km_k.labels_)
    print(f"k={k}: silhouette={sil:.3f}")


**Findings (Steps 18-19), reported honestly:**

With the redundancy removed, the two axes explain 62% and 23% of the
cross-province pattern (85% combined). PC1 loads positively on all six
subindexes (an "overall strength" axis, consistent with the earlier
version). **PC2 is dominated by `LTI` (Longevity & Type Index) and `EI`
(Environmental Impact Index) pulling in opposite directions**, not "feed
efficiency vs. conformation" as an earlier draft of this analysis
labeled it, `EI` itself combines Methane Efficiency, Feed Efficiency, and
Body Maintenance Requirements, not feed efficiency alone.

**The silhouette score prefers 2 clusters (0.305) over 3 (0.257),** which
this analysis has used throughout for interpretability. **With adequate
KMeans initialization (`n_init=100`), KMeans and hierarchical (Ward)
clustering agree perfectly at k=3 (Adjusted Rand Index = 1.000)** - an
earlier draft of this analysis used `n_init=10`, an insufficiently thorough
initialization that produced a different, unstable KMeans solution and a
much weaker apparent agreement (ARI=0.34); that result has been corrected.

**How this project's provincial grouping should be described.** Two
clusters receive the strongest silhouette support. A three-cluster
exploratory solution is also interpretable and, with adequate
initialization, agrees perfectly with Ward hierarchical clustering.
Still, with only 10 data points (provinces) going into the clustering,
this should be read as an exploratory grouping, not a definitive
classification. Manitoba stands apart from every other province in both
the 2-cluster and 3-cluster solutions, the most defensible specific claim
from this analysis.


## Step 20 - Within-animal type-trait dispersion

A different question from each province's between-animal homogeneity
(consistent LPI scores across its 400 animals, computed earlier from each
province's LPI standard deviation): here we ask whether *individual
animals* tend to be strong on some type traits and weak on others, and
whether that varies by province. Lower dispersion means an animal's `MS`,
`F&L`, `DS`, and `RP` values sit closer to each other, it measures
type-profile evenness, not overall conformation merit, an animal weak on
all four would also show low dispersion here.


In [ ]:
TYPE_TRAITS_ONLY = ["MS", "F&L", "DS", "RP"]  # Conf excluded: it is itself built from these four, including it would double-count
z_conformation = (clean[TYPE_TRAITS_ONLY] - clean[TYPE_TRAITS_ONLY].mean()) / clean[TYPE_TRAITS_ONLY].std()
clean["Type-Trait Dispersion (within animal)"] = z_conformation.std(axis=1)

dispersion_by_province = (
    clean.groupby("Province Name", observed=True)["Type-Trait Dispersion (within animal)"]
    .mean().round(3).sort_values()
)
dispersion_by_province


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
dispersion_by_province.plot(kind="bar", ax=ax, color="#2E5B4D")
ax.set_ylabel("Avg. within-animal type-trait dispersion (lower = more even, not necessarily better)")
ax.set_title("Within-animal type-trait dispersion, by province")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


**Findings:** this is a different question from each province's
between-animal homogeneity, Quebec led on that separate measure
(consistent LPI scores across its 400 animals), but is only mid-to-lower
here. **Nova Scotia and Saskatchewan show the lowest average dispersion
among the four standardized type traits**, with Prince Edward Island close
behind, while **Manitoba and Newfoundland & Labrador show the highest**.
This measures type-profile evenness, not overall conformation merit,
exactly as Step 20.2 makes explicit for the broader 6-subindex version of
this same question. `Conf` is excluded from this measure since it is
itself built from `MS`/`F&L`/`DS`/`RP`, including it would double-count
conformation rather than adding independent information.


## Step 20.1 - Does the same LPI mean the same cow?

Step 20 measured dispersion among four standardized type traits only. This step asks a
more direct question: since LPI is a weighted sum of 6 subindexes (`PI`
40%, `LTI` 32%, `HWI` 8%, `RI` 10%, `MI` 5%, `EI` 5%), two animals can
reach an identical LPI through very different combinations of those 6
components. This is demonstrated directly below, rather than reduced to a
single "balance" score.


In [ ]:
SUBINDEXES = ["PI", "LTI", "HWI", "RI", "MI", "EI"]
subindex_data = clean.dropna(subset=SUBINDEXES + ["LPI"]).copy()

# Bin LPI into narrow 10-point bands, and find the band with the most animals
subindex_data["LPI Band"] = (subindex_data["LPI"] / 10).round() * 10
band_counts = subindex_data["LPI Band"].value_counts()
biggest_band = band_counts.idxmax()
same_lpi_group = subindex_data[subindex_data["LPI Band"] == biggest_band]

print(f"Animals within 10 LPI points of each other (LPI~{biggest_band:.0f}): n={len(same_lpi_group)}")
print()
print("Spread (max - min) in each subindex, among these near-identical-LPI animals:")
print((same_lpi_group[SUBINDEXES].max() - same_lpi_group[SUBINDEXES].min()).sort_values(ascending=False))


In [ ]:
# A concrete pair: two animals with LPI within 1 point of each other,
# but the most different subindex profile found in this narrow band.
from itertools import combinations
import numpy as np

sample = same_lpi_group.sample(min(200, len(same_lpi_group)), random_state=42)
best_pair, best_diff = None, 0
for i, j in combinations(sample.index, 2):
    a, b = sample.loc[i], sample.loc[j]
    if abs(a["LPI"] - b["LPI"]) <= 1:
        diff = np.abs(a[SUBINDEXES].values - b[SUBINDEXES].values).sum()
        if diff > best_diff:
            best_diff, best_pair = diff, (a, b)

if best_pair:
    a, b = best_pair
    print("Two animals with (near-)identical LPI, very different subindex profiles:")
    print(pd.DataFrame([a[["Name", "Province Name", "LPI"] + SUBINDEXES],
                        b[["Name", "Province Name", "LPI"] + SUBINDEXES]]))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
if best_pair:
    a, b = best_pair
    x = np.arange(len(SUBINDEXES))
    width = 0.35
    ax.bar(x - width/2, a[SUBINDEXES].values, width, label=f"{a['Name']} (LPI={a['LPI']:.0f})", color="#2E5B4D")
    ax.bar(x + width/2, b[SUBINDEXES].values, width, label=f"{b['Name']} (LPI={b['LPI']:.0f})", color="#C9A66B")
    ax.set_xticks(x)
    ax.set_xticklabels(SUBINDEXES)
    ax.set_ylabel("Subindex value")
    ax.set_title("Same LPI, different cow: two animals, near-identical LPI, different subindex profiles")
    ax.legend()
plt.tight_layout()
plt.show()


**Findings:** among the largest group of animals sharing essentially
the same LPI (within 10 points of each other), the spread between the
strongest and weakest animal on any single subindex is large, often
several hundred points. The concrete pair shown above makes this direct:
two animals can carry the identical LPI while one is comparatively
stronger on `HWI`/`RI`/`EI` and the other stronger on `LTI`/`MI`. This is
expected given how LPI is constructed (a weighted sum where the two
largest weights, `PI` and `LTI`, total 72%): reaching a given LPI does not
require even performance across all 6 domains, strength in the
heavily-weighted domains can offset weakness in the lightly-weighted ones.
**A given LPI value does not by itself indicate a similar underlying
animal.**


## Step 20.2 - Cross-subindex dispersion, an exploratory diagnostic (not an official balance measure)

Building on Step 20.1's direct demonstration, this step summarizes how
evenly each animal's 6 subindexes sit relative to each other, using an
independently constructed, equal-domain diagnostic. **This is not part of
the official LPI formula and does not reproduce its economic or breeding
weights** (which favor `PI` and `LTI` heavily over the other four). Lower
dispersion means the 6 values are numerically similar to each other, it
does **not** by itself mean those values are favorable: an animal with all
6 subindexes far below average would also show low dispersion. This
measure should always be read alongside overall level (e.g., LPI itself),
never alone.


In [ ]:
z_subindex = (clean[SUBINDEXES] - clean[SUBINDEXES].mean()) / clean[SUBINDEXES].std()

# Evenness across the six standardized LPI subindexes (lower = more even, NOT "more balanced" or "better")
clean["Cross-Subindex Dispersion"] = z_subindex.std(axis=1)
# The single weakest standardized subindex for this animal (higher = no single domain is far below average)
clean["Weakest Standardized Subindex"] = z_subindex.min(axis=1)
# How many of the 6 subindexes sit more than 1 dataset SD below the dataset mean
# (a relative marker within this elite, LPI-selected group, not a biological or economic threshold)
clean["N Relatively Low Subindexes"] = (z_subindex < -1).sum(axis=1)
clean["No Subindex Below Dataset -1 SD"] = clean["N Relatively Low Subindexes"] == 0

balance_summary = clean.groupby("Province Name", observed=True).agg(
    MeanLPI=("LPI", "mean"),
    MeanDispersion=("Cross-Subindex Dispersion", "mean"),
    MeanWeakestSubindex=("Weakest Standardized Subindex", "mean"),
    PctNoRelativelyLowSubindex=("No Subindex Below Dataset -1 SD", "mean"),
).round(3).sort_values("MeanDispersion")
balance_summary["PctNoRelativelyLowSubindex"] = (balance_summary["PctNoRelativelyLowSubindex"] * 100).round(1)
balance_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
balance_summary["MeanDispersion"].plot(kind="bar", ax=ax, color="#2E5B4D")
ax.set_ylabel("Avg. cross-subindex dispersion (lower = more even, not necessarily better)")
ax.set_title("Cross-subindex dispersion by province (exploratory, equal-domain diagnostic)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
welch_balance = pg.welch_anova(data=clean, dv="Cross-Subindex Dispersion", between="Province Name")
print("Welch ANOVA, cross-subindex dispersion across provinces:")
print(welch_balance)
print()

gh_balance = pg.pairwise_gameshowell(data=clean, dv="Cross-Subindex Dispersion", between="Province Name")
n_sig = (gh_balance["pval"] <= 0.05).sum()
print(f"Games-Howell: {n_sig} of {len(gh_balance)} province pairs are significantly different on this measure.")


**Findings:** Welch ANOVA finds a statistically significant
difference in cross-subindex dispersion across provinces (p<0.001), and
province is associated with approximately 4.6% of the observed variance
in this constructed score (a small effect in practical terms). Games-Howell
finds a small majority of province pairs (26 of 45) are statistically
different on this specific measure, a less clean-cut separation than for
LPI itself. Saskatchewan and Nova Scotia show the lowest average dispersion
(most even across the 6 subindexes); Ontario shows the highest (least
even). **This says nothing on its own about which province's profile is
better** - the level table (`MeanLPI`, `MeanWeakestSubindex`,
`PctNoRelativelyLowSubindex`) needs to be read alongside dispersion, not
instead of it, exactly as demonstrated in Step 20.1.

**What this analysis does and does not establish:**

- **It tests whether provincial elite profiles differ in cross-subindex
  evenness. It does not determine whether a more uneven profile is
  economically harmful, biologically undesirable, or intentionally
  specialized** - that would require real herd outcomes (production,
  health events, longevity, profitability) this dataset doesn't have.
- **This metric weights all 6 domains equally, which the official LPI
  explicitly does not do** (`PI` and `LTI` alone are 72% of LPI's weight).
  It is offered as an alternative diagnostic of profile shape, a
  non-compensatory critique of what LPI's own weighting can hide, not an
  official measure of balance or evidence that any breeding objective has
  been violated.
- **Selection-induced correlation is a real limitation here.** Every
  animal in this dataset was included specifically because it has a high
  LPI, itself a weighted sum of these same 6 subindexes. Conditioning
  on a selected top-scoring group like this can induce apparent trade-offs
  among components that would be weaker or absent in an unrestricted
  population, an animal with a relatively weak value in one domain may
  need stronger values elsewhere just to remain in the top 400. This
  affects this dispersion measure and, more importantly, the `DS`-`HWI`
  and `DS`-`RI` associations reported elsewhere in this project: those
  should be read as high-confidence findings *within this selected
  dataset*, with uncertain generalizability to an unrestricted population.


## Step 20.5 - Is provincial genomic representation associated with elite-population LPI?

A more useful question than "which province wins" is: **do provinces with
more genomically tested animals in their elite population show stronger or
more homogeneous elite populations?** This uses the `GS` field, which
(unlike `LPI Code`) looks trustworthy across all 10 provinces - checked
directly below, not assumed.


In [ ]:
integration_rows = []
for province, group in clean.groupby("Province Name", observed=True):
    pct_any_genomic = group["Is Genomic"].mean() * 100
    integration_rows.append({
        "Province": province,
        "% Genomic Representation": round(pct_any_genomic, 1),
        "Mean LPI": round(group["LPI"].mean(), 1),
        "LPI SD": round(group["LPI"].std(), 1),
        "LPI CV%": round(group["LPI"].std() / group["LPI"].mean() * 100, 2),
    })

integration_df = pd.DataFrame(integration_rows).sort_values("% Genomic Representation", ascending=False)
integration_df


In [ ]:
r_lpi, p_lpi = stats.pearsonr(integration_df["% Genomic Representation"], integration_df["Mean LPI"])
rho_lpi, prho_lpi = stats.spearmanr(integration_df["% Genomic Representation"], integration_df["Mean LPI"])
print(f"All 10 provinces -> Pearson r={r_lpi:.3f} (p={p_lpi:.4f}), Spearman rho={rho_lpi:.3f} (p={prho_lpi:.4f})")

# LPI has a conventional scale and base, not a true meaningful zero, so the
# coefficient of variation (CV = SD/mean) isn't ideally suited to it. SD is
# used as the primary "consistency" measure here; CV is reported alongside
# only for continuity with earlier drafts, and both are checked to confirm
# they lead to the same conclusion.
r_sd, p_sd = stats.pearsonr(integration_df["% Genomic Representation"], integration_df["LPI SD"])
r_cv, p_cv = stats.pearsonr(integration_df["% Genomic Representation"], integration_df["LPI CV%"])
print(f"vs. LPI SD (consistency, primary measure): r={r_sd:.3f} (p={p_sd:.4f})")
print(f"vs. LPI CV% (consistency, secondary measure): r={r_cv:.3f} (p={p_cv:.4f})")

# Which province, if any, sits furthest from the trend line? Check residuals directly
# instead of assuming which point is "most extreme."
z = np.polyfit(integration_df["% Genomic Representation"], integration_df["Mean LPI"], 1)
integration_df["Predicted LPI"] = np.poly1d(z)(integration_df["% Genomic Representation"])
integration_df["Residual"] = (integration_df["Mean LPI"] - integration_df["Predicted LPI"]).round(1)
print()
print("Residual from trend line, sorted (most negative = furthest below what genomic rate predicts):")
print(integration_df[["Province", "% Genomic Representation", "Mean LPI", "Residual"]].sort_values("Residual"))


In [ ]:
# Leave-one-province-out: instead of picking a single province to exclude
# after seeing which one is the "worst fit," this drops each province in
# turn and reports the full range of resulting correlations, so the
# relationship's dependence on any one province can be judged directly.
loo_results = []
for excluded_province in integration_df["Province"]:
    subset = integration_df[integration_df["Province"] != excluded_province]
    r_loo, p_loo = stats.pearsonr(subset["% Genomic Representation"], subset["Mean LPI"])
    loo_results.append({"Province excluded": excluded_province, "r": round(r_loo, 3), "p": round(p_loo, 4)})

loo_df = pd.DataFrame(loo_results).sort_values("r")
print(f"Full sample (n=10): r={r_lpi:.3f}, p={p_lpi:.4f}")
print(f"Leave-one-out range: r from {loo_df['r'].min():.3f} to {loo_df['r'].max():.3f}, median r={loo_df['r'].median():.3f}")
print()
loo_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
colors = ["#B23A3A" if p == "Newfoundland and Labrador" else "#2E5B4D" for p in integration_df["Province"]]
ax.scatter(integration_df["% Genomic Representation"], integration_df["Mean LPI"], s=120, c=colors)
for _, row in integration_df.iterrows():
    ax.annotate(row["Province"], (row["% Genomic Representation"], row["Mean LPI"]), fontsize=9, xytext=(6, 6), textcoords="offset points")

x_line = np.linspace(integration_df["% Genomic Representation"].min(), integration_df["% Genomic Representation"].max(), 50)
ax.plot(x_line, np.poly1d(z)(x_line), color="#888780", linestyle="--", linewidth=1.5)

ax.set_xlabel("% of top-400 animals with a genomic test on file")
ax.set_ylabel("Mean LPI")
ax.set_title(f"Genomic representation vs. elite population strength (r={r_lpi:.2f}, p={p_lpi:.3f}, n=10, red = largest residual)")
plt.tight_layout()
plt.show()


**Findings:** across all 10 provinces, genomic representation is
associated with average LPI (r=0.73, p=0.016; Spearman rho=0.81, p=0.005,
confirming the relationship isn't driven by outliers in the raw values).
**Newfoundland & Labrador has the largest residual**: it has the
5th-highest genomic testing rate (85.75%) but sits about 207 points below
what that rate predicts, about 2.4 times the size of the next-largest
absolute residual (Ontario). The leave-one-province-out check above, dropping
each province in turn rather than only the one identified as furthest from
the trend, shows the correlation ranges from 0.66 to 0.95 (median 0.72) depending
on which single province is excluded. **The relationship is positive and
holds up regardless of which province is dropped, but its exact strength
is sensitive to that choice with only 10 data points**, which is a more
honest summary than reporting only the single exclusion that produces the
strongest result. This dataset cannot explain why Newfoundland & Labrador
combines relatively high genomic testing with a lower-than-expected score,
so it's reported here as a genuine, unresolved pattern, not excluded on
data-quality grounds. **Genomic representation is not associated with LPI
consistency**, using SD as the primary measure (r=0.13, not significant;
CV gives the same conclusion, r=0.01). LPI has a conventional scale and
base rather than a true zero, so SD is a more defensible dispersion measure
here than the coefficient of variation, though both agree in this case:
that specific claim, found in earlier drafts of this analysis, does not
hold with the corrected data and has been dropped.

**What this does and doesn't support:** provinces differ in genomic
representation, and that representation is associated with mean LPI in
these elite lists. This analysis used only `% Genomic Representation`, it
did not simultaneously use GEBV share, EBV share, average reliability, or
the full PA/GPA/EBV/GEBV distribution, so it does not by itself
demonstrate broader "information integration" or "composition" across
pedigree, genomics, and phenotype together. **This dataset cannot
determine why provinces differ in genomic representation** - it contains
no information about producer education, cost, service access, or
awareness, so no claim about producers "not understanding the value" of
any service is supported here.


## Step 20.6 - Verifying specific figures cited in the conclusions

A few specific numbers appear in the written conclusions without a
dedicated code cell computing them. This step computes each one directly,
using exactly the population described, so every number in this project
can be traced to a concrete calculation.


In [ ]:
# 1. Manitoba: % of its top-400 below breed average (0) on Dairy Strength
mb_ds = clean.loc[clean["Province Name"] == "Manitoba", "DS"]
pct_below_mb_ds = (mb_ds < 0).mean() * 100
print(f"Manitoba, % of top-400 below breed average on Dairy Strength: {pct_below_mb_ds:.1f}%")


In [ ]:
# 2. Fat and Protein: % of ALL animals below breed average (0), vs. Dairy Strength
pct_below_fat = (clean["Fat"] < 0).mean() * 100
pct_below_prot = (clean["Prot"] < 0).mean() * 100
pct_below_ds = (clean["DS"] < 0).mean() * 100
print(f"% of all 4,000 animals below breed average on Fat: {pct_below_fat:.2f}%")
print(f"% of all 4,000 animals below breed average on Protein: {pct_below_prot:.2f}%")
print(f"% of all 4,000 animals below breed average on Dairy Strength: {pct_below_ds:.2f}%")


In [ ]:
# 3. LPI trend by birth-year cohort
lpi_by_cohort = clean.groupby("Birth Year", observed=True)["LPI"].agg(["mean", "count"]).round(1)
print("Average LPI by birth-year cohort:")
print(lpi_by_cohort)


In [ ]:
# 4. Feed Efficiency (FE) vs. body size / conformation
fe_pairs = [("FE", "DS"), ("FE", "Conf")]
for a, b in fe_pairs:
    sub = clean.dropna(subset=[a, b])
    r, p = stats.pearsonr(sub[a], sub[b])
    print(f"{a} vs. {b}: r={r:.3f} (p={p:.4f}, n={len(sub)})")


**Findings, matched against the figures cited in the conclusions:**
Manitoba's share of animals below breed average on Dairy Strength, the
share of all animals below breed average on Fat and Protein, the LPI trend
across birth cohorts, and the Feed Efficiency correlations with body
size/conformation are all computed directly above from the full dataset,
confirming the specific figures cited elsewhere in this project.


## Step 21 - Reproducibility notes

- `RANDOM_SEED = 42` is used everywhere randomness appears (KMeans, PCA) so
  re-running this notebook produces identical results.
- Library versions are pinned exactly (`==`) in `requirements.txt`,
  captured via `pip freeze` from the environment used to build and execute
  this notebook, not just minimum compatible versions.
- The data loading and cleaning steps involve no randomness - the same
  input files always produce the same cleaned dataset.


## Final summary

- Loaded and consolidated 10 provincial files (4,000 animals total).
- Cleaned data types and made `Act.`/`GS` codes explicit.
- Calculated study-dataset and provincial averages, and genomic testing rates.
- Compared top animals by component (kg and %) and by conformation, by
  province, using only animals with their own EBV data for a fair comparison.
- Mapped conformation strengths/weaknesses by province with readable bar
  charts (no heatmaps).
- Checked correlation between production and conformation, with assumptions
  (normality, linearity, outliers) verified first: essentially no
  relationship found (all |r| < 0.3).
- Validated the LPI regression with VIF, residual/QQ plots, and a
  homoscedasticity test - found and addressed multicollinearity in `Conf`.
- Ran Welch ANOVA and Games-Howell (primary, given unequal provincial
  variances) plus classic ANOVA and Tukey HSD (secondary sensitivity check)
  across all 10 provinces, and 95% confidence intervals for each province's
  average LPI.
- Used PCA and two independent clustering methods to find natural province
  groupings.
- Measured within-animal type-trait dispersion, which describes profile
  evenness rather than conformation merit, a different question from
  between-animal homogeneity.
- Demonstrated directly that animals with near-identical LPI can have very
  different underlying subindex profiles (Step 20.1), since LPI is a
  weighted sum where `PI`+`LTI` alone total 72% of the weight, and measured
  cross-subindex dispersion as a separate, exploratory, equal-domain
  diagnostic (Step 20.2), not an official balance index: a real but small
  province-level effect (about 4.6% of variance), with a small majority of
  province pairs (26 of 45) statistically different on this measure.
- Compared PA vs. EBV animals (all 10 provinces now have a plausible
  `LPI Code` split): differences between mature animals were small overall,
  with only Conformation reaching statistical significance, while genomic
  testing status shows a much larger effect on estimate *reliability* than
  on the estimate itself.
- Tested whether information integration (genomic representation) associates
  with elite-population strength across provinces: yes, a moderate,
  robustness-checked correlation (r=0.73, p=0.016, n=10 - no provinces
  excluded), though based on only 10 provincial observations and sensitive
  in magnitude to Newfoundland & Labrador specifically.

### Known data limitations (keep these in mind for your conclusions)
- Each province file is a **top-400-by-LPI list**, not the full population.
- All averages in this notebook are **study-dataset averages** (from these
  4,000 records only), not Lactanet's official national average.
- This is a **snapshot in time**: Lactanet's evaluations are released
  periodically, and top-400 membership shifts as new births, phenotype,
  and genomic data arrive. Every animal in this dataset is confirmed
  unique (0 overlap across all 10 provinces), and all 10 provinces show a
  plausible `LPI Code` split.
- Age is approximate (birth year only, no exact date), and the "2+ years"
  filter assumes but does not confirm an animal has its own EBV data.
- `GS` (genomic tested) can't distinguish "never tested" from "result
  still pending," especially for the youngest animals.
